[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cCoolBoox/IFCD107-alumnado/blob/main/laboratorios/S05/S05_04_estadistica_descriptiva_ic.ipynb)

# S05_04 · Estadística con Python (1): descriptiva, distribuciones, Monte Carlo e intervalos de confianza
**Sprint 5 · Matemáticas y estadística en código · ⏱ 60 min**

## Qué vas a aprender
1. A **describir** una serie con pandas: media, mediana, dispersión, percentiles y gráficos.
2. A calcular probabilidades con **distribuciones** de SciPy (normal, binomial).
3. A responder preguntas con una **simulación de Monte Carlo**.
4. A construir e interpretar un **intervalo de confianza**.

## Contexto TurisData
La dirección del alojamiento pregunta: ¿cuál es su ocupación típica y cuánto varía?, ¿con qué frecuencia supera el
85 %?, ¿cuánto *overbooking* puede permitirse dadas las cancelaciones?, ¿con cuánta seguridad puede dar una cifra
de ocupación? Son preguntas de estadística. Datos sintéticos.

In [ ]:
# Preparación (ejecuta esta celda siempre la primera)
import os, urllib.request

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local de un archivo de datos del curso (lo descarga si no está)."""
    for carpeta in ("datos", "../datos", "../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

SEMILLA = 42
ocupacion_df = pd.read_csv(dato("ocupacion_diaria.csv"), parse_dates=["fecha"])
ocupacion_df["mes"] = ocupacion_df["fecha"].dt.month
reservas = pd.read_csv(dato("reservas_turisdata.csv"))
print(ocupacion_df.shape, reservas.shape)
ocupacion_df.head()

## 1 · Estadística descriptiva
- **Centro:** media (sensible a extremos) y mediana (el valor del medio).
- **Dispersión:** desviación típica (`std`, con `ddof=1` en pandas) y rango intercuartílico (Q3 − Q1).
- **Forma:** asimetría (`skew`). `describe()` da un resumen rápido.

In [ ]:
print(ocupacion_df["ocupacion_pct"].describe().round(2))
print("Asimetría:", round(ocupacion_df["ocupacion_pct"].skew(), 2))

### Ejercicio 1 · Resumen de la ocupación
Con la variable `serie` (ocupación diaria), guarda en `resumen` un diccionario con cuatro claves, todas redondeadas a 2 decimales:
`"media"`, `"mediana"`, `"desviacion"` y `"iqr"` (rango intercuartílico: percentil 75 menos percentil 25 con `quantile`).

In [ ]:
serie = ocupacion_df["ocupacion_pct"]      # la serie de ocupación diaria
# TODO: serie.mean(), serie.median(), serie.std() y serie.quantile(0.75) - serie.quantile(0.25)
...
assert resumen == {"media": 68.49, "mediana": 67.7, "desviacion": 9.61, "iqr": 13.6}, f"Revisa tus cifras: {resumen}"
print("Correcto:", resumen)

### Ejercicio 2 · Por meses
Con `groupby("mes")`, calcula `media_por_mes` (media de ocupación de cada mes) y guarda en `mes_max` el mes de mayor
ocupación media y en `mes_min` el de menor. Después se dibuja un diagrama de cajas por mes.

In [ ]:
# TODO: ocupacion_df.groupby("mes")["ocupacion_pct"].mean(); idxmax() e idxmin() dan el mes
...
assert len(media_por_mes) == 12, "Debe haber una media por cada uno de los 12 meses"
assert mes_max == 1, f"El mes de mayor ocupación media es enero (1) y tienes {mes_max}"
assert mes_min == 9, f"El mes de menor ocupación media es septiembre (9) y tienes {mes_min}"

ocupacion_df.boxplot(column="ocupacion_pct", by="mes", figsize=(9, 3.8))
plt.suptitle("")
plt.title("Ocupación diaria por mes (2023-2025)")
plt.xlabel("Mes")
plt.ylabel("Ocupación (%)")
plt.tight_layout()
plt.show()
print(f"Correcto: mayor ocupación en el mes {mes_max} ({media_por_mes[mes_max]:.1f} %), menor en el mes {mes_min} ({media_por_mes[mes_min]:.1f} %)")

**Lectura:** la ocupación es alta en invierno (turismo que huye del frío europeo) y baja en verano. Guardemos la
idea: la **temporada** influye mucho. Volverá en el siguiente cuaderno.

## 2 · Distribuciones con SciPy
Una **distribución** describe qué valores son más o menos probables.
- `norm(mu, sigma)`: la campana; `.cdf(x)` = P(X ≤ x), `.sf(x)` = P(X > x), `.ppf(p)` = el valor que deja a su izquierda una probabilidad p.
- `binom(n, p)`: número de éxitos en `n` intentos con probabilidad `p` (por ejemplo, cuántas reservas se cancelan de 20).

In [ ]:
mu, sigma = serie.mean(), serie.std()
print("P(ocupación > 85 %) según la normal:", round(stats.norm.sf(85, mu, sigma), 4))
print("Proporción real de días > 85 %:     ", round((serie > 85).mean(), 4))
print("Ocupación que se supera solo el 10 % de los días:", round(stats.norm.ppf(0.90, mu, sigma), 1))

### Ejercicio 3 · Probabilidad con la binomial
Un día llegan 20 reservas y cada una se cancela con probabilidad `p_cancelacion = 0.459` (la tasa global de
TurisData). Guarda en `p_10_o_mas` la probabilidad de que se cancelen **10 o más**. Pista: `stats.binom.sf(k, n, p)`
da P(X > k), así que P(X ≥ 10) = `sf(9, ...)`.

In [ ]:
p_cancelacion = 0.459
# TODO: stats.binom.sf(9, 20, p_cancelacion): probabilidad de superar 9 cancelaciones (es decir, 10 o más)
...
assert abs(p_10_o_mas - 0.4407) < 0.001, f"P(X ≥ 10) ≈ 0.4407 y tienes {p_10_o_mas:.4f}. Recuerda: sf(k) = P(X > k)"
print(f"Correcto: hay un {100 * p_10_o_mas:.1f} % de probabilidad de que se cancelen 10 o más de 20")

## 3 · Simulación de Monte Carlo
Cuando la matemática exacta es difícil, **simulamos** el proceso muchas veces y miramos con qué frecuencia ocurre
lo que nos interesa. Ejemplo: un hotel de **100 habitaciones** vende **170 reservas** (overbooking). Cada reserva
se presenta con probabilidad `1 − 0.459`. ¿Con qué probabilidad se presentan **más de 100** clientes (y hay que reubicar a alguien)?

In [ ]:
rng = np.random.default_rng(SEMILLA)
p_presentarse = 1 - 0.459
# Una simulación = 170 reservas, cada una se presenta (1) o no (0)
una_simulacion = rng.random(170) < p_presentarse
print("Se presentan en esta simulación:", una_simulacion.sum())

### Ejercicio 4 · Simular 10.000 días
Repite la simulación **10.000 veces** (sin bucles: `rng.random((10000, 170)) < p_presentarse`) y guarda en
`prob_reubicar` la proporción de simulaciones con **más de 100 presentados**. Compara con el valor exacto de la
binomial (`stats.binom.sf(100, 170, p_presentarse)`).

In [ ]:
# TODO: matriz (10000, 170) de True/False; suma por filas con axis=1; proporción de filas con suma > 100
...
exacta = stats.binom.sf(100, 170, p_presentarse)
assert presentados.shape == (10000,), "Debe haber un total de presentados por cada una de las 10000 simulaciones"
assert abs(prob_reubicar - exacta) < 0.02, f"La simulación ({prob_reubicar:.3f}) debería acercarse al valor exacto ({exacta:.3f})"
print(f"Correcto: Monte Carlo = {prob_reubicar:.3f} · exacto = {exacta:.3f}")

**Decisión de negocio:** con 170 reservas para 100 habitaciones, ~9 % de los días habría que reubicar a algún
cliente. ¿Es aceptable? Depende del coste de reubicar frente al de una habitación vacía. La estadística da el
número; la decisión es del cliente.

## 4 · Intervalos de confianza
La ocupación media de esta serie es una **estimación** con datos de 3 años. Un **intervalo de confianza del 95 %**
es un rango de valores plausibles para la media "verdadera": si repitiéramos el estudio muchas veces, el 95 % de
los intervalos construidos así contendría la media real. (No es "hay un 95 % de probabilidad de que la media esté aquí".)

Para una media: `media ± t · error_estándar`, con `error_estándar = std / √n`. SciPy lo da hecho:
`stats.t.interval(0.95, gl, loc=media, scale=error)`.

In [ ]:
n = len(serie)
error = stats.sem(serie)            # error estándar de la media
ic_total = stats.t.interval(0.95, n - 1, loc=serie.mean(), scale=error)
print(f"Media {serie.mean():.2f} · IC 95 %: ({ic_total[0]:.2f}, {ic_total[1]:.2f})")

### Ejercicio 5 · IC de una proporción
La **tasa de cancelación** de las 6.000 reservas es una proporción. Su IC aproximado del 95 % es
`p ± 1.96 · √(p (1 − p) / n)`. Calcula `p_hat`, `n_res` y `ic_tasa` (tupla `(inferior, superior)`, redondeada a 4 decimales).

In [ ]:
# TODO: p_hat = media de la columna cancelada; error = raiz(p(1-p)/n); tupla p - 1.96*error, p + 1.96*error
...
assert abs(p_hat - 0.459) < 1e-9 and n_res == 6000, "p_hat = 0.459 y n_res = 6000"
assert ic_tasa == (0.4464, 0.4716), f"El IC debería ser (0.4464, 0.4716) y tienes {ic_tasa}"
print(f"Correcto: tasa de cancelación {100 * p_hat:.1f} % (IC 95 %: {100 * ic_tasa[0]:.1f} % a {100 * ic_tasa[1]:.1f} %)")

## 5 · Mini-reto integrador: ¿enero y septiembre son realmente distintos?
Calcula el IC del 95 % de la ocupación media de **enero** (`ic_ene`) y de **septiembre** (`ic_sep`) con
`stats.t.interval` (cada uno con los datos de ese mes, tupla de 2 valores). Guarda en `se_solapan` un
booleano: ¿se solapan los dos intervalos? Interpreta el resultado.

In [ ]:
# TODO: filtra cada mes, calcula media y sem de ese mes y aplica stats.t.interval; se solapan si inferior_a <= superior_b y inferior_b <= superior_a
...
assert abs(ic_ene[0] - 77.48) < 0.05 and abs(ic_ene[1] - 80.18) < 0.05, f"IC de enero ≈ (77.48, 80.18) y tienes {ic_ene}"
assert abs(ic_sep[0] - 58.86) < 0.05 and abs(ic_sep[1] - 61.27) < 0.05, f"IC de septiembre ≈ (58.86, 61.27) y tienes {ic_sep}"
assert se_solapan is False, "Los intervalos de enero y septiembre no se solapan"
print(f"Correcto. Enero: ({ic_ene[0]:.1f}, {ic_ene[1]:.1f}) · Septiembre: ({ic_sep[0]:.1f}, {ic_sep[1]:.1f}) → no se solapan: la diferencia no parece casualidad.")

## 6 · Para reflexionar
1. La normal daba un 4,3 % de días por encima del 85 % y en la realidad fueron un 5,4 %. ¿Qué te dice sobre usar la normal como aproximación?
2. ¿Cómo cambiaría el IC de la ocupación media si tuviéramos 10 años de datos en lugar de 3? ¿Y si la variabilidad fuera mayor?
3. ¿Qué diferencia hay entre "el IC no se solapa" y "la diferencia es importante para el negocio"?

## 7 · Qué has aprendido
- `describe`, media/mediana, desviación, IQR y diagramas de caja por grupo.
- `scipy.stats`: `cdf`, `sf`, `ppf` de la normal y la binomial.
- Monte Carlo: simular muchas veces para estimar una probabilidad y comprobarla con la teoría.
- Un intervalo de confianza acompaña a una estimación con su incertidumbre.